## Silver transformations

In [0]:
USE CATALOG workouts_demo;
USE SCHEMA silver;

CREATE OR REPLACE TABLE workouts_demo.silver.activities AS
WITH base AS (
  SELECT
    CAST(b.activity_id AS BIGINT)                                        AS activity_id,
    CAST(b.activity_type AS STRING)                                      AS sport,
    CAST(b.elapsed_time5 AS DOUBLE)                                      AS elapsed_time_s,
    CAST(b.moving_time AS DOUBLE)                                        AS moving_time_s,
    CAST(b.distance17 AS DOUBLE)                                         AS distance_m,
    CAST(b.elevation_gain AS DOUBLE)                                     AS elevation_m,
    -- ← Parse and derive month
    month(to_timestamp(b.activity_date, 'd MMM yyyy, HH:mm:ss'))         AS activity_month_int,
    COALESCE(b._metadata.file_modification_time, current_timestamp())    AS _ingested_at
  FROM workouts_demo.bronze.activities_raw b
  WHERE b.activity_type IN ('Run','Walk','Ride','Nordic Ski')
),
dedup AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY activity_id
      ORDER BY _ingested_at DESC
    ) AS rn
  FROM base
)
SELECT 
  activity_id, 
  activity_month_int, 
  elevation_m,
  sport, 
  elapsed_time_s, 
  moving_time_s, 
  distance_m,
  CASE
    WHEN moving_time_s > 0 AND distance_m IS NOT NULL
      THEN distance_m / moving_time_s      -- m/s
  END AS avg_speed_mps
FROM dedup
WHERE rn = 1 AND activity_id IS NOT NULL AND elevation_m IS NOT NULL;
